In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, Layout
from gekko import GEKKO
%matplotlib inline

In [6]:
def model(x):
    return np.array([x[0]-x[0]*x[1]-0.4*x[0],
                     -x[1]+x[0]*x[1]-0.2*x[1]])
def plot_pp(a,b):
    t0 = 0
    tmax = 12
    sol = solve_ivp(lambda t, x:model(x), [t0, tmax], np.array([a,b]), t_eval=np.linspace(t0, tmax, 100))
    plt.subplot(1,2,1)
    plt.plot(sol.t,sol.y[0], label = 'Prey')
    plt.plot(sol.t,sol.y[1], label = 'Predator')
    plt.xlabel("time")
    plt.ylabel("population")
    plt.title("Solution of the ODE system")
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(sol.y[1],sol.y[0], label = 'Prey', color = "r")
    plt.rcParams["figure.figsize"]=10,5
    plt.title("Phase portrait of the solution")
    plt.xlabel("predator population")
    plt.ylabel("prey population")

style = {'description_width': 'initial'}
layout = Layout(width = "400px")
slid1 = widgets.FloatSlider(description = 'Initial Prey Population $a$ $\quad$',min=0, max=1, step=0.1, value=0.5,style = style, layout = layout)
slid2 = widgets.FloatSlider(description = 'Initial Predator Population $b$',min=0, max=1, step=0.1, value=0.7, style = style, layout = layout)
interact(plot_pp, a= slid1 ,b=slid2)


interactive(children=(FloatSlider(value=0.5, description='Initial Prey Population $\\quad$', layout=Layout(wid…

<function __main__.plot_pp(a, b)>

In [7]:
def u(t):
    if 0 <= t <= 2:
        return 0
    elif 2 < t <= 4.5:
        return 1
    elif 4.5 < t <= 7.5:
        return 0.5
    else:
        return 0
vu = np.vectorize(u, otypes=[np.float64])

def model_control_ex(t, x):
    return np.array([x[0] - x[0] * x[1] - 0.4 * x[0] * u(t), -x[1] + x[0] * x[1] - 0.2 * x[1] * u(t)])


def plot_pp(a, b):
    t0 = 0
    tmax = 12
    sol = solve_ivp(lambda t, x: model_control_ex(t, x), [t0, tmax], np.array([a, b]),
                    t_eval=np.linspace(t0, tmax, 100))
    plt.subplot(1, 2, 1)
    plt.plot(sol.t, sol.y[0], label='Prey')
    plt.plot(sol.t, sol.y[1], label='Predator')
    plt.xlabel("time")
    plt.ylabel("population")
    plt.title("Solution of the ODE system")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(sol.y[1], sol.y[0], label='Prey', color="r")
    plt.rcParams["figure.figsize"] = 10, 5
    plt.title("Phase portrait of the solution")
    plt.xlabel("predator population")
    plt.ylabel("prey population")


style = {'description_width': 'initial'}
layout = Layout(width="400px")
slid1 = widgets.FloatSlider(description='Initial Prey Population $a$ $\quad$', min=0, max=1, step=0.1, value=0.5,
                            style=style, layout=layout)
slid2 = widgets.FloatSlider(description='Initial Predator Population $b$', min=0, max=1, step=0.1, value=0.7, style=style,
                            layout=layout)
interact(plot_pp, a=slid1, b=slid2)


interactive(children=(FloatSlider(value=0.5, description='Initial Prey Population $\\quad$', layout=Layout(wid…

<function __main__.plot_pp(a, b)>

In [5]:
def plot_pp(a,b):
    m = GEKKO(remote = True)
    n = 100
    T = 12
    m.time = np.linspace(0,T,n)
    y0 = 1

    x1 = m.Var(value = a)
    x2 = m.Var(value = b)
    x3 = m.Var(value = (a-y0)**2 + (b-y0)**2)
    u = m.Var(value = 0, lb = 0, ub = 1)
    p = np.zeros(n)
    p[-1] = T

    m.Equation(x1.dt()==x1-x1*x2-0.4*x1*u)
    m.Equation(x2.dt()==-x2+x1*x2-0.2*x1*u)
    m.Equation(x3.dt()==(x1-y0)**2+(x2-y0)**2)

    m.Obj(x3)
    m.options.IMODE = 6 # optimal control mode

    m.solve(disp=False)
    plt.subplot(1,2,1)
    plt.plot(m.time,x1.value, label = 'Prey')
    plt.plot(m.time,x2.value, label = 'Predator')
    plt.xlabel("time")
    plt.ylabel("population")
    plt.title("Solution to the control problem with $y_0$ =%.1f" %y0)
    plt.legend()
    plt.subplot(1,2,2)
    plt.plot(m.time,u.value, color = "r")
    plt.rcParams["figure.figsize"]=10,5
    plt.title("Control function values")
    plt.xlabel("time")
    plt.ylabel("control value")


style = {'description_width': 'initial'}
layout = Layout(width = "400px")
slid1 = widgets.FloatSlider(description = 'Initial Prey Population $a$ $\qquad$',min=0.4, max=0.8, step=0.05, value=0.5,style = style, layout = layout)
slid2 = widgets.FloatSlider(description = 'Initial Predator Population $b$',min=0.4, max=0.8, step=0.05, value=0.5, style = style, layout = layout)
interact(plot_pp, a= slid1 ,b=slid2)

interactive(children=(FloatSlider(value=0.5, description='Initial Prey Population $a$ $\\quad$', layout=Layout…

<function __main__.plot_pp(a, b)>